# ChurnSense — Model Building

## Objective

The objective of this notebook is to build and compare multiple machine learning models for customer churn prediction.

We will build:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting
5. XGBoost

We will use:
- Preprocessing pipelines
- Stratified K-Fold cross-validation
- ROC-AUC for model comparison
- Classification reports for evaluation

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    classification_report
)

In [3]:
import sys
sys.path.append("..")

from src.preprocessing import load_data, prepare_data, create_preprocessor

In [4]:
df = pd.read_csv(r"C:\Users\PARTH\Downloads\WA_Fn-UseC_-Telco-Customer-Churn.csv")

X, y, numeric_features, categorical_features = prepare_data(df)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 20)
y shape: (7043,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (5634, 20)
Testing shape: (1409, 20)


In [6]:
preprocessor = create_preprocessor(
    numeric_features,
    categorical_features
)

In [7]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    )
}

In [8]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models["Logistic Regression"])
])

In [9]:
decision_tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models["Decision Tree"])
])

random_forest_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models["Random Forest"])
])

gradient_boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models["Gradient Boosting"])
])

xgboost_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models["XGBoost"])
])

In [10]:
pipelines = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": decision_tree_pipeline,
    "Random Forest": random_forest_pipeline,
    "Gradient Boosting": gradient_boosting_pipeline,
    "XGBoost": xgboost_pipeline
}

In [11]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [12]:
logistic_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [13]:
decision_tree_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [14]:
random_forest_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
gradient_boosting_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
xgboost_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
cv_results = {}

for name, pipeline in pipelines.items():
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )
    
    cv_results[name] = scores.mean()
    
    print(f"{name}:")
    print(f"CV AUC: {scores.mean():.4f}")
    print()

Logistic Regression:
CV AUC: 0.8465

Decision Tree:
CV AUC: 0.6906

Random Forest:
CV AUC: 0.8272

Gradient Boosting:
CV AUC: 0.8483

XGBoost:
CV AUC: 0.8236



In [18]:
comparison_df = pd.DataFrame(
    list(cv_results.items()),
    columns=["Model", "CV_AUC"]
)

comparison_df = comparison_df.sort_values(
    by="CV_AUC",
    ascending=False
).reset_index(drop=True)

comparison_df

,Model,CV_AUC
0,Gradient Boosting,0.848255
1,Logistic Regression,0.846479
2,Random Forest,0.827158
3,XGBoost,0.823589
4,Decision Tree,0.690553


In [19]:
test_results = {}

for name, pipeline in pipelines.items():
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_pred_proba)
    test_results[name] = auc
    
    print(f"{name}: Test AUC = {auc:.4f}")

Logistic Regression: Test AUC = 0.8419
Decision Tree: Test AUC = 0.6970
Random Forest: Test AUC = 0.8219
Gradient Boosting: Test AUC = 0.8464
XGBoost: Test AUC = 0.8221


In [20]:
test_comparison_df = pd.DataFrame(
    list(test_results.items()),
    columns=["Model", "Test_AUC"]
)

test_comparison_df

,Model,Test_AUC
0,Logistic Regression,0.841889
1,Decision Tree,0.696968
2,Random Forest,0.821873
3,Gradient Boosting,0.846408
4,XGBoost,0.822147


In [21]:
for name, pipeline in pipelines.items():
    y_pred = pipeline.predict(X_test)

    print("=" * 60)
    print(name)
    print("=" * 60)
    print(classification_report(y_test, y_pred))

Logistic Regression
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.65      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409

Decision Tree
              precision    recall  f1-score   support

           0       0.84      0.87      0.85      1035
           1       0.59      0.53      0.55       374

    accuracy                           0.78      1409
   macro avg       0.71      0.70      0.70      1409
weighted avg       0.77      0.78      0.77      1409

Random Forest
              precision    recall  f1-score   support

           0       0.83      0.91      0.86      1035
           1       0.65      0.47      0.54       374

    accuracy                           0.79      1409
   macro avg       0.74      0.69      0.70      1409
weighted avg       0.78   